# M2: run the classical baselines

This notebook runs noisy input, BM3D and NLM on the fixed SIDD validation split.
It does not train a CNN. New output folders keep the archived measurements intact.
In Colab, select a CPU runtime, run the cells in order and upload
`Mennatalla_M2_Run_Data.zip` when asked. Locally, first prepare `data/m2` using the
README instructions and open this notebook from the repository or notebooks folder.

The full run processes 120 crops and may take many minutes. Set `LIMIT = 1` for
a quick smoke test; a smoke-test result is not the full M2 table.

In Colab, the run cell connects Drive by default so per-crop snapshots survive
a runtime disconnect. You can set `SAVE_TO_DRIVE = False` to use temporary
storage instead; download the ZIP before disconnecting in that case.


In [ ]:
from pathlib import Path
import subprocess
import sys
import importlib.util

IN_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'scripts/run_m2_baselines.py').is_file()), None)
if ROOT is None:
    if not IN_COLAB:
        raise RuntimeError('Open the notebook from the cloned repository.')
    ROOT = Path('/content/c17_m2_baselines')
    if not ROOT.exists():
        subprocess.check_call(['git', 'clone', '--branch', 'menna-m2-review',
                               'https://github.com/hosseldin/self-supervised-image-denoising.git', str(ROOT)])
    if not (ROOT / 'scripts/run_m2_baselines.py').is_file():
        raise RuntimeError('The existing checkout has no M2 runner; use a fresh Colab runtime.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements-m2.txt')])
print('Repository:', ROOT)


In [ ]:
import uuid

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Choose only Mennatalla_M2_Run_Data.zip.')
    archive = ROOT / 'data' / 'Mennatalla_M2_Run_Data.zip'
    archive.parent.mkdir(exist_ok=True)
    archive.write_bytes(next(iter(uploaded.values())))
    DATA = ROOT / 'data' / ('m2_' + uuid.uuid4().hex[:8])
    subprocess.check_call([sys.executable, str(ROOT / 'scripts/run_m2_baselines.py'),
                           'prepare', '--validation-zip', str(archive), '--output', str(DATA)])
else:
    DATA = ROOT / 'data/m2'
    if not (DATA / 'manifest.csv').is_file():
        raise FileNotFoundError('Prepare data/m2 using the README before running this cell.')
print('Data:', DATA)


In [ ]:
from datetime import datetime, timezone

LIMIT = None  # None evaluates all 120 crops; use 1 for a quick smoke test.
SAVE_TO_DRIVE = True
OUTPUT_BASE = ROOT / 'runs'
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_BASE = Path('/content/drive/MyDrive/C17_M2_Baseline_Runs')
    OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
    token = uuid.uuid4().hex
    probe = OUTPUT_BASE / ('write_check_' + token + '.txt')
    probe.write_text(token)
    if probe.read_text() != token:
        raise RuntimeError('Drive write/read check failed; do not start the run.')
    probe.unlink()
OUTPUT = OUTPUT_BASE / ('m2_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6])
command = [sys.executable, str(ROOT / 'scripts/run_m2_baselines.py'), 'run',
           '--data', str(DATA), '--output', str(OUTPUT), '--threads', '2']
if LIMIT is not None:
    command += ['--limit', str(LIMIT)]
subprocess.check_call(command)
print('Completed output:', OUTPUT)


In [ ]:
from IPython.display import Image, display

print((OUTPUT / 'comparison.csv').read_text())
display(Image(filename=str(OUTPUT / 'validation_comparison.png')))
display(Image(filename=str(OUTPUT / 'before_after.png')))


In [ ]:
import shutil

archive_path = Path(shutil.make_archive(str(OUTPUT), 'zip', root_dir=OUTPUT))
print('Results ZIP:', archive_path)
if IN_COLAB:
    files.download(str(archive_path))
print('Also save the executed notebook with File > Download > Download .ipynb.')
